# L05: Three approaches to one allocation problem

**Day 1.** Deterministic, heuristic and stochastic optimization on one convex allocation problem: gradient descent and its learning rate, exhaustive grid search, simulated annealing, and the sensitivity of the answer to its constraints.

- Reads: checkpoints/L03_findings.json, data/crew_capacity.json
- Writes: work/L05_allocation.json

**Before you start.** Open this notebook from the **Notebooks** folder of the class workspace, next to the **data** and **checkpoints** folders, and run the first two cells.

**How this lab runs.** Work through the cells in order on your own. Each one says what to do with it: **RUN** supplied code, **PREDICT** before running, **DECIDE** by filling a blank, or **WRITE** an answer in words. When most of the room is done, we discuss the answers as a class, then walk through the worked solution together.

### Step 1 · RUN

In [ ]:
# RUN · setup: finds the class files and checks that today's are there. Run it first, every time.
from pathlib import Path
# The class files sit in the folder holding "data" and "checkpoints": this notebook's folder or the one above it.
# (On a local install, a folder named AIDSA in your user folder also works.)
_here = Path.cwd()
ROOT = next((p for p in [_here, *_here.parents, Path.home() / "AIDSA"] if (p / "data").is_dir() and (p / "checkpoints").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Class files not found: open this notebook from the Notebooks folder of the class workspace.")
DATA, CHECKPOINTS, WORK = ROOT / "data", ROOT / "checkpoints", ROOT / "work"
NEEDED = ["checkpoints/L03_findings.json",
          "data/crew_capacity.json"]
missing = [name for name in NEEDED if not (ROOT / name).exists()]
if missing:
    raise FileNotFoundError("Not in the workspace yet: " + ", ".join(missing) + ". Ask your instructor: today's files are not in the workspace yet.")
WORK.mkdir(exist_ok=True)
SOURCE = "checkpoint"  # change to "mine" to use your own saved work where it exists
SEED = 2026
import numpy as np
rng = np.random.default_rng(SEED)

def input_path(name):
    """Where this lab reads an earlier lab's output: your own work folder if you chose "mine" and the file is there,
    otherwise the supplied checkpoint."""
    mine = WORK / name
    return mine if SOURCE == "mine" and mine.exists() else CHECKPOINTS / name

print("Found the class files in:", ROOT, "| start from:", SOURCE)

### Step 2 · RUN

In [ ]:
# RUN · helper functions: the tested course helpers this lab uses. Run it once, after the setup cell.
# You do not need to read them to do the lab; each is explained where the lab uses it.
import itertools
import numpy as np


def delay_curve(x, n, r, k, m):
    """Expected delayed departures per week for allocation x (the objective F).

    Lab L05.
    """
    x, n, r, k, m = (np.asarray(v, dtype=float) for v in (x, n, r, k, m))
    return float(np.sum(n * r * np.exp(-k * (x - m))))


def gradient(x, n, r, k, m):
    """Partial derivatives of the objective with respect to each type's crews.

    Lab L05.
    """
    x, n, r, k, m = (np.asarray(v, dtype=float) for v in (x, n, r, k, m))
    return -k * n * r * np.exp(-k * (x - m))


def project(x, m, capacity):
    """Nearest allocation (in Euclidean distance) with sum equal to capacity and x_i >= m_i.

    Projection onto a shifted simplex, by the sort-and-threshold method.

    Lab L05.
    """
    x, m = np.asarray(x, dtype=float), np.asarray(m, dtype=float)
    y, spare = x - m, capacity - m.sum()
    u = np.sort(y)[::-1]
    cum = np.cumsum(u) - spare
    idx = np.arange(1, y.size + 1)
    rho = np.nonzero(u - cum / idx > 0)[0][-1]
    return m + np.maximum(y - cum[rho] / (rho + 1), 0.0)


def gradient_descent(x0, n, r, k, m, capacity, learning_rate, iterations=200):
    """Projected gradient descent. Bounded: always exactly `iterations` steps.

    Returns (final allocation, objective value after every step).

    Lab L05.
    """
    x = project(x0, m, capacity)
    history = []
    for _ in range(iterations):
        x = project(x - learning_rate * gradient(x, n, r, k, m), m, capacity)
        history.append(delay_curve(x, n, r, k, m))
    return x, history


def grid_allocate(n, r, k, m, capacity):
    """Exhaustive search over whole-crew allocations. Returns (best allocation, value, number evaluated).

    Bounded by the problem size: every way of placing the spare crews above the minimums.

    Lab L05.
    """
    m = np.asarray(m, dtype=int)
    spare = int(capacity - m.sum())
    best, best_value, evaluated = None, np.inf, 0
    for extra in itertools.product(range(spare + 1), repeat=m.size - 1):
        if sum(extra) > spare:
            continue
        x = m + np.array(list(extra) + [spare - sum(extra)])
        value = delay_curve(x, n, r, k, m)
        evaluated += 1
        if value < best_value:
            best, best_value = x, value
    return best, best_value, evaluated


def anneal(n, r, k, m, capacity, iterations=3000, start_temp=0.5, seed=2026):
    """Simulated annealing over whole-crew allocations: move one crew between two types at random,
    always accept an improvement, accept a worsening with probability exp(-increase / temperature).

    Bounded (fixed iterations) and seeded. Returns (best allocation, value, objective after each step).

    Lab L05.
    """
    rng = np.random.default_rng(seed)
    m = np.asarray(m, dtype=int)
    x = m.copy()
    x[0] += int(capacity - m.sum())                     # start with every spare crew on one type
    current = delay_curve(x, n, r, k, m)
    best, best_value, history = x.copy(), current, []
    for step in range(iterations):
        temp = start_temp * (1 - step / iterations) + 1e-6
        i, j = rng.choice(m.size, size=2, replace=False)
        if x[i] <= m[i]:
            history.append(current)
            continue
        candidate = x.copy()
        candidate[i] -= 1
        candidate[j] += 1
        value = delay_curve(candidate, n, r, k, m)
        if value < current or rng.random() < np.exp(-(value - current) / temp):
            x, current = candidate, value
            if value < best_value:
                best, best_value = candidate.copy(), value
        history.append(current)
    return best, best_value, history


def sensitivity(n, r, k, m, capacity, learning_rate=10.0, iterations=400):
    """Value of one more crew, and of relaxing each type's minimum by one crew.

    Each figure is the drop in expected delayed departures per week, from re-solving the
    continuous problem with gradient descent. A minimum that is not binding is worth zero.

    Lab L05.
    """
    m = np.asarray(m, dtype=float)
    start = np.full(m.size, capacity / m.size)
    base = gradient_descent(start, n, r, k, m, capacity, learning_rate, iterations)[1][-1]
    more = gradient_descent(start, n, r, k, m, capacity + 1, learning_rate, iterations)[1][-1]
    relax = []
    for i in range(m.size):
        m2 = m.copy()
        m2[i] -= 1
        # r is the rate at the minimum, so shift it to keep the same curve when the minimum moves.
        r2 = np.asarray(r, dtype=float).copy()
        r2[i] *= np.exp(np.asarray(k, dtype=float)[i])
        relax.append(base - gradient_descent(start, n, r2, k, m2, capacity, learning_rate, iterations)[1][-1])
    return {"objective": base, "one_more_crew": base - more, "relax_minimum": relax}

print("Helper functions ready")

## Part 1: Formulation

**Specification.** Turn the Day 1 delay rates into an optimization problem: split 24 maintenance crews across the four
aircraft types, each above its minimum, to minimise expected delayed departures per week.

### Step 3 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
import json

import matplotlib.pyplot as plt
import numpy as np

findings = json.loads(input_path("L03_findings.json").read_text(encoding="utf-8"))
crews = json.loads((DATA / "crew_capacity.json").read_text(encoding="utf-8"))
types = crews["aircraft_types"]
n = np.array([findings["departures_by_type"][t] / 52 for t in types])    # departures per week
r = np.array([findings["delay_rate_by_type"][t] for t in types])         # delay rate at the minimum crew level
k = np.array(crews["crew_effect"])                                        # effect of each extra crew
m = np.array(crews["minimum_crews"], dtype=float)                         # minimum crews per type
C = crews["capacity_crews"]


def F(x):
    """Objective: expected delayed departures per week for allocation x."""
    return delay_curve(x, n, r, k, m)


even = project(np.full(4, C / 4), m, C)
print(f"{C} crews, minimums {m.astype(int).tolist()} (spare {int(C - m.sum())})")
print(f"even split {even.round(1).tolist()}: {F(even):.3f} delayed departures per week")

fig, ax = plt.subplots(figsize=(6, 3.2))
grid = np.linspace(0, 8, 50)
for i, t in enumerate(types):
    ax.plot(m[i] + grid, n[i] * r[i] * np.exp(-k[i] * grid), label=t)
ax.set_xlabel("crews assigned")
ax.set_ylabel("expected delayed departures / week")
ax.legend()
plt.show()

### Step 4 · WRITE

**WRITE · your interpretation**. Why is this problem convex, and what does convexity promise about the answer gradient descent finds?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

## Part 2: Gradient descent

**Specification.** Choose the direction of the update, run gradient descent at three learning rates, and carry forward
the one whose curve shows it converged.

### Step 5 · DECIDE

In [ ]:
# DECIDE · DP-L05-1 · which way the update steps.
# Replace ____ with your choice. The choices are taught on the slide "Gradient descent: step downhill, repeatedly".
# The update steps from x by learning_rate times the gradient g, then projects back onto the constraints.
def descend(learning_rate, iterations=200):
    """Projected gradient descent from the even split; returns (allocation, objective history)."""
    x = even.copy()
    history = []
    for _ in range(iterations):                                  # bounded: always 200 steps
        g = gradient(x, n, r, k, m)
        x = project(x ____ learning_rate * g, m, C)
        history.append(F(x))
    return x, history

### Step 6 · PREDICT

**PREDICT · before you run the next cell**. The next cell runs 200 steps at learning rates of 0.05, 10 and 300. Which one will settle, which will
stall, and which will never settle? Say what you expect each curve to look like.

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

### Step 7 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
runs = {lr: descend(lr) for lr in (10, 300, 0.05)}
fig, ax = plt.subplots(figsize=(6, 3.2))
for lr, (x, history) in runs.items():
    ax.plot(history, label=f"learning rate {lr}")
    print(f"learning rate {lr:>5}: allocation {x.round(2).tolist()}, final {history[-1]:.3f}, "
          f"range over last 20 steps {np.ptp(history[-20:]):.3f}")
ax.set_xlabel("iteration")
ax.set_ylabel("expected delayed departures / week")
ax.legend()
plt.show()

### Step 8 · DECIDE

In [ ]:
# DECIDE · DP-L05-2 · which learning rate to carry forward.
# Replace ____ with your choice. The choices are taught on the slide "The learning rate: what each setting does".
# Read the three curves above, then choose the run whose allocation the rest of the lab should use.
LEARNING_RATE = ____
x_gd = runs[LEARNING_RATE][0]
print(f"carrying forward learning rate {LEARNING_RATE}: {F(x_gd):.3f} delayed departures per week")

### Step 9 · WRITE

**WRITE · your interpretation**. Describe what went wrong at each bad learning rate, and how you would recognise it on a real problem
where the minimum is unknown.

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

## Part 3: Grid search and annealing

**Specification.** Solve the same problem two more ways: exhaustively over whole crews, and by simulated annealing.
Compare the three answers.

### Step 10 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
x_grid, v_grid, evaluated = grid_allocate(n, r, k, m, C)
x_sa, v_sa, sa_history = anneal(n, r, k, m, C, iterations=3000, seed=SEED)
print(f"gradient descent (continuous): {x_gd.round(2).tolist()} -> {F(x_gd):.3f}")
print(f"grid search ({evaluated} allocations): {x_grid.tolist()} -> {v_grid:.3f}")
print(f"simulated annealing (3,000 steps): {x_sa.tolist()} -> {v_sa:.3f}")

### Step 11 · WRITE

**WRITE · your interpretation**. When would you choose each method? Consider problem size, whether the problem is convex, and whether
the answer must be in whole crews.

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

## Part 4: Sensitivity

**Specification.** Ask what the answer is worth at the margin: one more crew, and one fewer crew required for each type.

### Step 12 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
sens = sensitivity(n, r, k, m, C)
print(f"optimum: {sens['objective']:.3f} delayed departures per week")
for t, x_opt, value in zip(types, x_gd, sens["relax_minimum"]):
    print(f"{t:<8} allocated {x_opt:5.2f} (minimum {int(m[types.index(t)])}); "
          f"relaxing its minimum by one crew saves {value:.3f}")

### Step 13 · DECIDE

In [ ]:
# DECIDE · DP-L05-3 · which number answers the commander.
# Replace ____ with your choice. The choices are taught on the slide "Sensitivity: what the answer is worth at the margin".
# The commander asks what one more crew is worth. Choose that quantity from the sensitivity results.
one_more_crew = sens["____"]
print(f"one more crew prevents {one_more_crew:.3f} delayed departures per week "
      f"(about {one_more_crew * 52:.0f} a year)")

### Step 14 · WRITE

**WRITE · your interpretation**. Which constraint binds, and what would you tell a commander asking for one more crew?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

### Step 15 · RUN

In [ ]:
# RUN · saves this part's output to work/.
result = {"types": types, "capacity": C, "gradient_descent": x_gd.round(3).tolist(),
          "grid_search": x_grid.tolist(), "annealing": x_sa.tolist(), "objective": round(sens["objective"], 4),
          "one_more_crew": round(one_more_crew, 4),
          "relax_minimum": [round(v, 4) for v in sens["relax_minimum"]]}
(WORK / "L05_allocation.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
print("saved work/L05_allocation.json")